[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C60_Edge_Deployment_Consistency_Course/03_int8/03_int8_calibration.ipynb)

# 03 · INT8 校准与精度恢复（KL 熵校准 / 校准集分布不匹配 / 敏感层与混合精度）

目标：**从零实现 TensorRT 的熵校准算法**，并用可控实验量化三件事——
不同校准算法的差别、校准集选错的代价、以及混合精度能救回多少。全程纯 numpy。

本 notebook 你会亲手实现：
1. **对称量化的基本件**：`quantize/dequantize`、SQNR、ENOB、per-tensor vs **per-channel**
2. **三种校准算法**：MinMax / Percentile / **完整的 KL 熵校准**（2048-bin 直方图、
   参考分布构造、候选分布的「量化-反量化」重建、KL 扫描）
3. 三者在含离群值分布上的对比 —— 包括一个**反直觉结论**：
   按全张量 SQNR 评判，MinMax 反而赢
4. **校准集分布不匹配实验**：只用白天数据校准 → 夜间裁剪率 50%、下游准确率 0.79 → 0.51
5. **采样策略对比**：「训练集头 500 张」 vs 随机 vs **分层采样**
6. **逐层敏感度分析**（孤立量化 / 恢复到 FP16 两个方向，并证明它们不等价）
   + **含 reformat 代价的混合精度方案搜索**
7. 检测专属：**IoU 对坐标误差的尺寸依赖**、多尺度 concat 共用 scale 的灾难

> 心智模型：**校准算法决定「保留哪部分信息」，校准集决定「哪部分信息存在」。
> 后者错了，前者再对也没用。**

## 1 · 对称量化的基本件

TensorRT 的约定：对称（zero-point = 0）、有符号、$q\in[-127,127]$、$S=T/127$。

$\mathrm{ENOB}=(\mathrm{SQNR}_{\mathrm{dB}}-1.76)/6.02$ —— 把「信噪比」翻译成「实际用上了几位」。

In [ ]:
import numpy as np, math
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

N_LEVELS = 127          # TRT 用 [-127,127]，放弃 -128 以保证严格对称

def quantize(x, thr, n=N_LEVELS):
    """对称量化：返回整数码 q 与 scale"""
    s = thr / n
    return np.clip(np.round(np.asarray(x) / s), -n, n), s

def quant_dequant(x, thr, n=N_LEVELS):
    q, s = quantize(x, thr, n)
    return q * s, s

def sqnr_db(x, xq):
    x = np.asarray(x, dtype=float); xq = np.asarray(xq, dtype=float)
    sig = float((x ** 2).sum()); noi = float(((x - xq) ** 2).sum())
    return 10 * math.log10(sig / max(noi, 1e-30))

def enob(sqnr):
    return (sqnr - 1.76) / 6.02

# —— 手算校验 ——
thr = 127.0                                   # 于是 scale = 1.0，量化码就是四舍五入
xq, s = quant_dequant([0.0, 1.0, 2.4, 2.6, -3.5, 200.0], thr)
print('scale =', s, ' 量化-反量化结果 =', xq)
assert s == 1.0
assert np.allclose(xq, [0., 1., 2., 3., -4., 127.])   # 200 被 clip 到 127；-3.5 银行家舍入到 -4
print()

x = np.abs(rng.normal(0, 1.0, 100000))
for t in [x.max(), 4.0, 3.0, 2.0]:
    xq, s = quant_dequant(x, t)
    d = sqnr_db(x, xq)
    print('thr=%6.3f  step=%.5f  SQNR=%6.2f dB  ENOB=%.2f bit  裁剪率=%.3f%%'
          % (t, s, d, enob(d), 100 * np.mean(x > t)))
print()
print('✅ 半正态分布上，阈值从 max 降到 3σ：SQNR 反而上升 —— 因为步长变小的收益')
print('   超过了裁掉 0.3% 尾巴的代价。**这就是所有校准算法在权衡的东西。**')

In [ ]:
# per-tensor vs per-channel：depthwise 卷积的权重是最典型的场景
C = 32
W = rng.normal(0, 1, (C, 25))                                  # 32 个通道，每通道 5x5 核
chan_gain = np.exp(rng.normal(0, 1.1, (C, 1)))                 # **通道间尺度差异极大**
W = W * chan_gain
print('各通道 max|w| 的最小/中位/最大 = %.4f / %.4f / %.4f  （相差 %.0f 倍）'
      % (np.abs(W).max(1).min(), np.median(np.abs(W).max(1)),
         np.abs(W).max(1).max(), np.abs(W).max(1).max() / np.abs(W).max(1).min()))

Wq_pt, _ = quant_dequant(W, np.abs(W).max())                   # per-tensor：一个 scale 走天下
s_pc = np.abs(W).max(axis=1, keepdims=True) / N_LEVELS         # per-channel：每个输出通道一个
Wq_pc = np.clip(np.round(W / s_pc), -N_LEVELS, N_LEVELS) * s_pc

d_pt, d_pc = sqnr_db(W, Wq_pt), sqnr_db(W, Wq_pc)
print('per-tensor  SQNR = %6.2f dB  (ENOB %.2f bit)' % (d_pt, enob(d_pt)))
print('per-channel SQNR = %6.2f dB  (ENOB %.2f bit)' % (d_pc, enob(d_pc)))
worst = int(np.argmin(np.abs(W).max(1)))
print('最弱通道（#%d）的 SQNR: per-tensor %.2f dB -> per-channel %.2f dB'
      % (worst, sqnr_db(W[worst], Wq_pt[worst]), sqnr_db(W[worst], Wq_pc[worst])))
assert d_pc > d_pt + 6, (d_pt, d_pc)
assert sqnr_db(W[worst], Wq_pc[worst]) > sqnr_db(W[worst], Wq_pt[worst]) + 10
print()
print('✅ **权重 per-channel 是免费的**（scale 与求和变量无关，可提到 INT32 累加之外），')
print('   所以它不是「优化」，是 depthwise / 分组卷积的硬要求。')
print('⚠️  而激活的 per-channel scale 落在求和号<内>，会破坏 INT32 累加 —— 做不到。')
print('   SmoothQuant 的思路正是：把激活的通道差异用对角矩阵「搬」到权重上去。')

## 2 · 造一个真实感的激活分布，先跑两个便宜的算法

规则（贴近 ReLU 后的卷积激活）：
- 主体：**半正态** $|N(0,1)|$，20 万个样本
- 离群：**0.02%**（40 个）落在 30–60 —— 来自过曝像素 / 某个特殊通道 / 数值不稳定的归一化

**注意这个 0.02%：它决定了 Percentile 的分位点该怎么选。**

In [ ]:
def make_activation(n_bulk=200000, n_out=40, lo=30.0, hi=60.0, seed=0):
    r = np.random.default_rng(seed)
    bulk = np.abs(r.normal(0, 1.0, n_bulk))
    out = r.uniform(lo, hi, n_out)
    return np.concatenate([bulk, out])

def minmax_threshold(x):
    return float(np.abs(x).max())

def percentile_threshold(x, p=99.9):
    return float(np.percentile(np.abs(x), p))

def clip_rate(x, thr):
    return float(np.mean(np.abs(x) > thr))

def effective_levels(x, thr, q=99.0):
    """主体（q 分位以内）占用了 127 个电平里的几个"""
    return float(np.percentile(np.abs(x), q) / (thr / N_LEVELS))

X = make_activation()
print('样本数 %d   max=%.2f   99%%=%.3f   99.9%%=%.3f   99.99%%=%.3f'
      % (len(X), X.max(), np.percentile(X, 99), np.percentile(X, 99.9), np.percentile(X, 99.99)))
print('离群比例 = %.4f%%  （%d 个）' % (100 * np.mean(X > 20), int((X > 20).sum())))
assert abs(np.mean(X > 20) - 40 / 200040) < 1e-6
print()
print('MinMax        阈值 %.3f' % minmax_threshold(X))
print('Percentile99.9  阈值 %.3f' % percentile_threshold(X, 99.9))
print('Percentile99.99 阈值 %.3f   <- **分位点落进了离群值里**' % percentile_threshold(X, 99.99))
assert percentile_threshold(X, 99.99) > 10 * percentile_threshold(X, 99.9)
print()
print('⚠️  离群比例是 0.02%，而 99.99% 分位切掉的是 0.01% —— 分位点直接落在离群 population 内。')
print('   **「分位点越高越保守」是错的。分位点必须跟离群比例匹配，而离群比例逐层不同。**')

## 3 · 完整实现 TensorRT 的 KL 熵校准

对每个候选截断点 `i`（从 128 扫到 2048）：

1. **参考分布 P**：`P = hist[:i]`，然后 `P[i-1] += sum(hist[i:])`
   —— 被截掉的质量并进最后一个 bin，这就是「clip 到阈值」的分布语义
2. **候选分布 Q**：把 `[0,i)` 均分成 128 组求和（= 量化），
   再**按组内非零 bin 的个数**摊回去（= 反量化），**原本是 0 的 bin 保持 0**
3. 归一化 + 平滑，算 `KL(P‖Q)`

取 KL 最小的 `i*`，**threshold = (i* + 0.5) × bin 宽**。

In [ ]:
def _smooth(p, eps=1e-4):
    """把一点点质量从非零 bin 挪给零 bin，避免 KL 因 Q 出现 0 而发散"""
    p = np.asarray(p, dtype=np.float64)
    z = (p == 0); nz = ~z
    if not nz.any():
        return None
    out = p.copy()
    out[z] = eps
    out[nz] = out[nz] - eps * z.sum() / nz.sum()
    return np.maximum(out, eps * 1e-3)          # 数值保护，防止减成负数

def quantize_hist(h, n_levels=128):
    """把长度为 i 的直方图压成 n_levels 个电平，再摊回长度 i。
       这是熵校准里「量化-反量化」的分布版本。"""
    h = np.asarray(h, dtype=np.float64)
    i = len(h)
    nm = i // n_levels                                   # 每组多少个 bin（余数并入最后一组）
    starts = np.arange(n_levels) * nm
    sums = np.add.reduceat(h, starts)                    # 每组求和  = 量化
    nzc = np.add.reduceat((h > 0).astype(np.float64), starts)
    gid = np.repeat(np.arange(n_levels), np.diff(np.append(starts, i)))
    q = np.where(h > 0,                                  # **零 bin 保持零**
                 np.where(nzc[gid] > 0, sums[gid] / np.maximum(nzc[gid], 1.0), 0.0),
                 0.0)
    return q

def kl_divergence(p, q):
    p = np.asarray(p, dtype=np.float64); q = np.asarray(q, dtype=np.float64)
    p = p / p.sum(); q = q / q.sum()
    m = p > 0
    return float(np.sum(p[m] * np.log(p[m] / q[m])))

def entropy_threshold(x, n_bins=2048, n_levels=128, return_curve=False):
    a = np.abs(np.asarray(x).ravel())
    amax = float(a.max())
    hist, _ = np.histogram(a, bins=n_bins, range=(0.0, amax))
    hist = hist.astype(np.float64)
    best_i, best_kl, curve = n_levels, math.inf, []
    for i in range(n_levels, n_bins + 1):
        h = hist[:i]
        p = h.copy()
        p[-1] += hist[i:].sum()                          # ① 离群值并进最后一个 bin
        q = quantize_hist(h, n_levels)                   # ② 量化-反量化
        q[p == 0] = 0.0
        ps, qs = _smooth(p), _smooth(q)                  # ③ 平滑
        if ps is None or qs is None:
            continue
        kl = kl_divergence(ps, qs)
        curve.append((i, kl))
        if kl < best_kl:
            best_kl, best_i = kl, i
    thr = (best_i + 0.5) * (amax / n_bins)
    return (thr, best_i, best_kl, curve) if return_curve else thr

# —— 手算校验 quantize_hist ——
h = np.array([4, 0, 2,  0, 6, 0,  3, 3, 0,  1, 0, 0], dtype=float)   # 12 个 bin -> 4 个电平
q = quantize_hist(h, n_levels=4)
print('h =', h.astype(int))
print('q =', q)
# 组1 [4,0,2] 和=6 非零=2 -> [3,0,3]；组2 [0,6,0] 和=6 非零=1 -> [0,6,0]
# 组3 [3,3,0] 和=6 非零=2 -> [3,3,0]；组4 [1,0,0] 和=1 非零=1 -> [1,0,0]
assert np.allclose(q, [3, 0, 3, 0, 6, 0, 3, 3, 0, 1, 0, 0]), q
assert abs(q.sum() - h.sum()) < 1e-12, '总质量必须守恒'
assert np.all((h == 0) == (q == 0)), '零 bin 必须保持零'
assert abs(kl_divergence(h + 1e-9, h + 1e-9)) < 1e-12, 'KL(P||P) = 0'
print('✅ quantize_hist / kl_divergence 手算通过')

In [ ]:
# —— 三个 sanity check：算法在「没有离群」时应该几乎不裁剪 ——
u = rng.uniform(0, 1, 100000)
t_u = entropy_threshold(u)
print('均匀分布  : KL 阈值 %.4f   max %.4f   -> 比值 %.3f' % (t_u, u.max(), t_u / u.max()))
assert 0.97 < t_u / u.max() < 1.05, t_u

g = np.abs(rng.normal(0, 1, 200000))
t_g = entropy_threshold(g)
print('半正态分布: KL 阈值 %.4f   max %.4f   -> 比值 %.3f' % (t_g, g.max(), t_g / g.max()))
assert 0.85 < t_g / g.max() < 1.05, t_g
print()
print('✅ **KL 校准只在有离群时才裁剪。**干净的分布上它给出接近 max 的阈值 ——')
print('   这说明它不是「无脑收紧」，而是从分布形状里把该不该截断算出来的。')

In [ ]:
# —— 主实验：四种算法在含离群分布上的完整对比 ——
t_mm  = minmax_threshold(X)
t_p9  = percentile_threshold(X, 99.9)
t_p99 = percentile_threshold(X, 99.99)
t_kl, best_i, best_kl, curve = entropy_threshold(X, return_curve=True)
print('KL 扫描：最优 bin i* = %d / 2048，KL = %.3e，阈值 = %.4f' % (best_i, best_kl, t_kl))
print()

inl = np.abs(X) <= np.percentile(np.abs(X), 99.0)          # 「主体」= 99% 分位以内
hdr = '%-18s %8s %9s %10s %10s %9s %9s'
print(hdr % ('算法', '阈值T', '步长S', '主体电平数', '主体SQNR', 'ENOB', '裁剪率'))
res = {}
for name, t in [('MinMax', t_mm), ('Percentile 99.9', t_p9),
                ('Percentile 99.99', t_p99), ('Entropy (KL)', t_kl)]:
    xq, s = quant_dequant(X, t)
    d_all, d_in = sqnr_db(X, xq), sqnr_db(X[inl], xq[inl])
    res[name] = dict(thr=t, sqnr_all=d_all, sqnr_in=d_in,
                     lv=effective_levels(X, t), clip=clip_rate(X, t))
    print('%-18s %8.3f %9.5f %10.1f %9.2f dB %8.2f b %8.3f%%'
          % (name, t, s, effective_levels(X, t), d_in, enob(d_in), 100 * clip_rate(X, t)))

assert res['Entropy (KL)']['thr'] < res['MinMax']['thr'] / 5
assert res['Entropy (KL)']['sqnr_in'] > res['MinMax']['sqnr_in'] + 15
assert res['Entropy (KL)']['lv'] > 8 * res['MinMax']['lv']
assert res['Percentile 99.99']['sqnr_in'] < res['Percentile 99.9']['sqnr_in'] - 15
print()
print('⚠️  **MinMax 把一个 8 位量化器变成了 %.1f 位。**' % enob(res['MinMax']['sqnr_in']))
print('   主体只占 %.1f 个电平（127 个里），而 KL 让它占 %.0f 个。'
      % (res['MinMax']['lv'], res['Entropy (KL)']['lv']))
print('⚠️  Percentile 99.99 比 99.9 差了整整 %.1f dB —— 分位点选错比 MinMax 还糟。'
      % (res['Percentile 99.9']['sqnr_in'] - res['Percentile 99.99']['sqnr_in']))

In [ ]:
# —— 反直觉的一格：如果用**全张量** SQNR 评判，谁赢？——
print('%-18s %12s %12s' % ('算法', '全张量 SQNR', '主体 SQNR'))
for name in ['MinMax', 'Percentile 99.9', 'Percentile 99.99', 'Entropy (KL)']:
    r = res[name]
    print('%-18s %9.2f dB %9.2f dB' % (name, r['sqnr_all'], r['sqnr_in']))

assert res['MinMax']['sqnr_all'] > res['Entropy (KL)']['sqnr_all'] + 5
assert res['MinMax']['sqnr_in'] < res['Entropy (KL)']['sqnr_in'] - 15
print()
print('⚠️  **按全张量 SQNR，MinMax 赢；按主体 SQNR，KL 赢 20+ dB。**')
print('   原因：KL 裁掉了那 0.02% 的离群值，而离群值的平方误差极大，直接主导了全张量 MSE。')
print()
print('   这说明「哪个校准算法好」这个问题本身依赖于**你认为张量的哪部分信息重要**：')
print('   · MSE 准则会保留极值（它们的平方误差大）')
print('   · 信息（KL）准则会牺牲极值（它们的概率质量小）')
print('   **而这个立场对不对，只能由下游任务指标回答，不能由张量级指标回答。**')
print('   -> 所以校准算法的选择必须落到**分桶评测**上，这不是形式主义。')

In [ ]:
# —— 离群规模扫描：三种算法各自怎么反应 ——
print('%-12s %10s %10s %10s %10s %11s' %
      ('离群占比', '离群幅度', 'MinMax', 'P99.9', 'KL', 'KL/MinMax'))
ratios = {}
for n_out, lo, hi in [(1, 4.0, 4.01), (10, 10.0, 20.0), (40, 30.0, 60.0),
                      (200, 30.0, 60.0), (1000, 30.0, 60.0)]:
    Xo = make_activation(n_out=n_out, lo=lo, hi=hi, seed=5)
    tm, tp, tk = minmax_threshold(Xo), percentile_threshold(Xo, 99.9), entropy_threshold(Xo)
    ratios[n_out] = tk / tm
    print('%-12s %10.1f %10.3f %10.3f %10.3f %11.3f'
          % ('%.3f%%' % (100.0 * n_out / len(Xo)), hi, tm, tp, tk, tk / tm))
assert ratios[1] > 0.8, '几乎没有离群时，KL 不该裁剪'
assert ratios[40] < 0.15, '0.02% 的极端离群 -> KL 把阈值压到 MinMax 的十分之一以下'
assert ratios[1000] > 0.9, '离群占到 0.5% 时，它们已经不算离群了，KL 会保留'
print()
print('✅ 三段行为，三个结论：')
print('   · 没有离群         -> KL ≈ MinMax（它不是无脑收紧）')
print('   · 极少量极端离群   -> KL 只有 MinMax 的 %.0f%%（阈值跟着**主体**走）'
      % (100 * ratios[40]))
print('   · 离群多到 0.5%    -> KL 又回到 MinMax 附近')
print('     **因为占 0.5% 质量的东西已经不是「离群」，是分布的一部分。**')
print()
print('⚠️  顺带一个工程结论：**MinMax 的结果不可复现** —— 阈值完全由最极端的')
print('   那一个样本决定，换一批校准数据就变。KL 与 Percentile 都稳定得多。')

## 4 · 校准集分布不匹配：把危害量化出来

场景：一个「特征 → 分类头」的最小检测器切片。

- **白天**：激活 `relu(M[y] + N(0,1))`，最大值约 6
- **夜间**：同样的特征，但整体乘一个 **3–12 倍的增益**
  （夜间相机自动提高 ISO 与曝光；交通标志的反光膜被车灯照射后接近饱和）
  → 最大值约 57

关键：**夜间的 float 精度和白天一样好**（正缩放不改变 argmax）。
所以接下来看到的一切掉点，**全部来自校准集选错**。

In [ ]:
D_FEAT, K_CLS = 64, 8
M_PROTO = np.abs(rng.normal(0, 1.0, (K_CLS, D_FEAT))) * 0.9      # 每类的原型向量

def gen_scene(n, kind, seed):
    r = np.random.default_rng(seed)
    y = r.integers(0, K_CLS, n)
    f = np.maximum(M_PROTO[y] + r.normal(0, 1.0, (n, D_FEAT)), 0)
    if kind == 'night':
        f = f * r.uniform(3.0, 12.0, (n, 1))       # 夜间高增益：整体放大
    elif kind == 'tunnel':
        f = f * r.uniform(4.0, 15.0, (n, 1))       # 隧道出口：更极端
    return f, y

def head_acc(f, y):
    return float((np.argmax(np.asarray(f) @ M_PROTO.T, axis=1) == y).mean())

day,   y_day   = gen_scene(4000, 'day',   1)
night, y_night = gen_scene(1000, 'night', 2)
print('白天激活 max=%.2f  99.9%%=%.2f' % (day.max(), np.percentile(day, 99.9)))
print('夜间激活 max=%.2f  99.9%%=%.2f   <- 大一个数量级' % (night.max(), np.percentile(night, 99.9)))
print('**FP 精度**：白天 %.3f   夜间 %.3f   （夜间本身没有更难）'
      % (head_acc(day, y_day), head_acc(night, y_night)))
assert abs(head_acc(day, y_day) - head_acc(night, y_night)) < 0.05
print('✅ 基线确认：夜间的困难**完全来自量化**，不来自任务本身。')

In [ ]:
CAL_SETS = {
    '❌ 只用白天 512 张':        np.concatenate([gen_scene(512, 'day', 11)[0]]),
    '✅ 白天+10%夜间（分层）':   np.concatenate([gen_scene(460, 'day', 13)[0],
                                                gen_scene(52,  'night', 14)[0]]),
    '⚠️ 只用夜间 512 张':        np.concatenate([gen_scene(512, 'night', 12)[0]]),
}
print('%-26s %8s %9s %9s %9s %9s %10s' %
      ('校准集', '阈值T', 'SQNR白天', 'SQNR夜间', 'acc白天', 'acc夜间', '夜间裁剪率'))
out = {}
for name, cal in CAL_SETS.items():
    t = percentile_threshold(cal, 99.9)
    dq, _ = quant_dequant(day, t)
    nq, _ = quant_dequant(night, t)
    out[name] = dict(thr=t, sd=sqnr_db(day, dq), sn=sqnr_db(night, nq),
                     ad=head_acc(dq, y_day), an=head_acc(nq, y_night),
                     clip=clip_rate(night, t))
    r = out[name]
    print('%-26s %8.2f %8.1fdB %8.1fdB %9.3f %9.3f %9.1f%%'
          % (name, r['thr'], r['sd'], r['sn'], r['ad'], r['an'], 100 * r['clip']))

bad, good, nite = out['❌ 只用白天 512 张'], out['✅ 白天+10%夜间（分层）'], out['⚠️ 只用夜间 512 张']
assert bad['an'] < good['an'] - 0.2, (bad['an'], good['an'])
assert bad['clip'] > 0.3, bad['clip']
assert good['ad'] > bad['ad'] - 0.02
assert bad['sd'] > good['sd'] > nite['sd'], '校准集越偏夜间，白天的分辨率被稀释得越多'
print()
print('⚠️  **只用白天校准：夜间准确率 %.3f -> %.3f（掉 %.1f 个点），裁剪率 %.0f%%。**'
      % (good['an'], bad['an'], 100 * (good['an'] - bad['an']), 100 * bad['clip']))
print('   同样的模型、同样的算法、同样的 512 张图 —— **只是选图的方式不同**。')
print()
print('⚠️  反过来「只用夜间」也有代价：白天 SQNR 从 %.1f dB 掉到 %.1f dB'
      % (bad['sd'], nite['sd']))
print('   （ENOB %.1f bit -> %.1f bit）。本例中分类头的间距够大所以准确率没塌，'
      % (enob(bad['sd']), enob(nite['sd'])))
print('   但**回归头没有这种容错**（下一节会看到）。所以要按部署分布分层，不是走极端。')

In [ ]:
# —— 采样策略：「训练集头 500 张」到底错在哪 ——
N_SEQ, FR_PER_SEQ = 500, 20                       # 500 个采集片段，每段 20 帧
SCENES = ['day', 'dusk', 'night', 'rain', 'tunnel']
SCENE_P = [0.62, 0.12, 0.14, 0.08, 0.04]          # 部署分布
r0 = np.random.default_rng(42)
seq_scene = list(r0.choice(SCENES, N_SEQ, p=SCENE_P))
for i in range(80):                               # **前 80 个片段是同一次白天采集**
    seq_scene[i] = 'day'                          #   （现实里数据集就是按采集时间排的）

GAIN = {'day': (1.0, 1.0), 'dusk': (1.6, 2.4), 'night': (3.0, 12.0),
        'rain': (1.2, 1.8), 'tunnel': (4.0, 15.0)}
frames = []
for si, sc in enumerate(seq_scene):
    rr = np.random.default_rng(1000 + si)
    lo, hi = GAIN[sc]
    g = rr.uniform(lo, hi)                        # 同一片段内增益几乎不变 -> 帧间高度冗余
    y = rr.integers(0, K_CLS, FR_PER_SEQ)
    f = np.maximum(M_PROTO[y] + rr.normal(0, 1.0, (FR_PER_SEQ, D_FEAT)), 0) * g
    for k in range(FR_PER_SEQ):
        frames.append((si, sc, f[k]))
print('数据集：%d 个片段 × %d 帧 = %d 帧' % (N_SEQ, FR_PER_SEQ, len(frames)))

def take(idx):
    return ([frames[i][0] for i in idx], [frames[i][1] for i in idx],
            np.stack([frames[i][2] for i in idx]))

rs = np.random.default_rng(9)
strategies = {}
strategies['❌ 头 500 帧'] = list(range(500))
strategies['△ 随机 500 帧'] = list(rs.choice(len(frames), 500, replace=False))
# 分层：每个场景按部署比例配额，且**保底每个场景 40 帧**，每片段最多取 1 帧
quota = {sc: max(40, int(500 * p)) for sc, p in zip(SCENES, SCENE_P)}
seq_first = {}
for i, (si, sc, _) in enumerate(frames):
    seq_first.setdefault((sc, si), i)
strat = []
for sc in SCENES:
    cand = [i for (s, _), i in seq_first.items() if s == sc]
    rs.shuffle(cand)
    strat += cand[:quota[sc]]
strategies['✅ 分层 + 每片段 1 帧'] = strat

print()
print('%-24s %7s %10s %10s %9s %9s %9s' %
      ('采样策略', '帧数', '独立片段数', '夜间+隧道占比', '阈值T', 'acc夜间', '夜间裁剪率'))
picked = {}
for name, idx in strategies.items():
    seqs, scs, feats = take(idx)
    t = percentile_threshold(feats, 99.9)
    nq, _ = quant_dequant(night, t)
    tail = np.mean([s in ('night', 'tunnel') for s in scs])
    picked[name] = dict(nseq=len(set(seqs)), tail=tail, thr=t,
                        acc=head_acc(nq, y_night), clip=clip_rate(night, t))
    p = picked[name]
    print('%-24s %7d %10d %11.1f%% %9.2f %9.3f %8.1f%%'
          % (name, len(idx), p['nseq'], 100 * tail, t, p['acc'], 100 * p['clip']))

h, rnd, st = picked['❌ 头 500 帧'], picked['△ 随机 500 帧'], picked['✅ 分层 + 每片段 1 帧']
assert h['nseq'] == 25, h['nseq']                       # 500 帧 / 20 帧每段 = 25 段
assert h['tail'] == 0.0, '头 500 帧全是白天'
assert h['acc'] < st['acc'] - 0.15
assert st['tail'] > rnd['tail'], '分层采样的尾部场景覆盖应高于随机'
assert st['nseq'] > rnd['nseq'] * 0.9
print()
print('⚠️  **「头 500 帧」的三重错误：**')
print('   ① 只有 %d 个独立片段（500 帧的信息量 ≈ 25 张图）' % h['nseq'])
print('   ② 尾部场景（夜间/隧道）覆盖率 0%% —— 阈值只有 %.2f，夜间裁剪率 %.0f%%'
      % (h['thr'], 100 * h['clip']))
print('   ③ 现实里它还带着训练增强（本实验没模拟，但那是第三个独立错误）')
print('   随机采样能解决 ①②，**分层采样还能保证稀有场景有下限配额**（隧道只占 4%，')
print('   随机 500 帧只给约 20 帧，统计上不稳）。')

## 5 · 逐层敏感度分析与混合精度

造一个 6 层的玩具网络，其中 **L1 与 L3 带「离群通道」**（3 个输出通道的权重放大 28 倍，
模拟 Transformer / 深层 CNN 里常见的 outlier channel）。
激活故意用 **MinMax 校准**（最脆弱的那种），好让量化误差足够大、现象足够清楚。

两个扫描方向：
- **孤立量化**：只把第 i 层设成 INT8，其余 FP16 → 「量化这层要付多少」
- **恢复到 FP16**：全 INT8，只把第 i 层放回 FP16 → 「救这层能拿回多少」

In [ ]:
D_NET, K_NET, L_NET = 48, 10, 6
rn = np.random.default_rng(3)
NET_W = []
for i in range(L_NET):
    Wl = rn.normal(0, 1.0 / np.sqrt(D_NET), (D_NET, D_NET))
    if i in (1, 3):
        idx = rn.choice(D_NET, 3, replace=False)
        Wl[:, idx] *= 28.0                      # **离群通道**
    NET_W.append(Wl)
NET_OUT = rn.normal(0, 1.0 / np.sqrt(D_NET), (D_NET, K_NET))
XN = np.maximum(rn.normal(0, 1, (3000, D_NET)), 0)

def qw_perchannel(W, n=N_LEVELS):
    s = np.abs(W).max(axis=0, keepdims=True) / n
    return np.clip(np.round(W / s), -n, n) * s

# 校准激活阈值（MinMax）
_h, ACT_THR = XN, []
for i in range(L_NET):
    _h = np.maximum(_h @ NET_W[i], 0)
    ACT_THR.append(float(np.abs(_h).max()))
print('各层激活阈值(MinMax):', ' '.join('%.1f' % t for t in ACT_THR))

def net_forward(x, bits):
    h = x
    for i in range(L_NET):
        W = qw_perchannel(NET_W[i]) if bits[i] == 8 else NET_W[i]
        h = np.maximum(h @ W, 0)
        if bits[i] == 8:
            h, _ = quant_dequant(h, ACT_THR[i])
    return h @ NET_OUT

REF = np.argmax(net_forward(XN, [16] * L_NET), axis=1)
def net_acc(bits):
    return float((np.argmax(net_forward(XN, bits), axis=1) == REF).mean())

ACC_FP16, ACC_INT8 = net_acc([16] * L_NET), net_acc([8] * L_NET)
print('全 FP16 = %.4f   全 INT8 = %.4f   -> 掉 %.1f 个点'
      % (ACC_FP16, ACC_INT8, 100 * (ACC_FP16 - ACC_INT8)))
assert ACC_FP16 == 1.0
assert ACC_INT8 < 0.85, ACC_INT8
print()
print('%6s %14s %16s' % ('层', '孤立量化的代价', '从全INT8恢复的收益'))
iso, restore = [], []
for i in range(L_NET):
    b1 = [16] * L_NET; b1[i] = 8
    b2 = [8] * L_NET;  b2[i] = 16
    c = ACC_FP16 - net_acc(b1)
    g = net_acc(b2) - ACC_INT8
    iso.append((c, i)); restore.append((g, i))
    print('%6s %14.4f %16.4f%s' % ('L%d' % i, c, g, '   <- 恢复反而略降' if g < 0 else ''))

rank_iso = [i for _, i in sorted(iso, reverse=True)]
rank_res = [i for _, i in sorted(restore, reverse=True)]
print()
print('孤立量化排序（最敏感在前）:', ' '.join('L%d' % i for i in rank_iso))
print('恢复收益排序（最值得救在前）:', ' '.join('L%d' % i for i in rank_res))
assert rank_iso != rank_res, '两个方向给出的排序应当不同'
assert min(g for g, _ in restore) < 0.01
print()
print('⚠️  **两个方向不等价**，因为量化误差在层间相互作用：')
print('   某层的误差可能被后一层放大，也可能被后面的 ReLU 吸收，甚至与别层的误差部分抵消。')
print('   （表里出现「恢复某层反而略降」正是抵消的表现 —— 这不是 bug。）')
print('   工程结论：**做混合精度方案时用「恢复」方向的排序，它更贴近最终配置；')
print('   而且任何方案都必须整体验证，不能把逐层收益加总。**')

In [ ]:
# —— 混合精度方案搜索：把 reformat 代价放进目标函数 ——
import itertools
T_INT8, T_FP16, T_REFORMAT = 1.0, 1.9, 0.2      # ms/层，以及每个精度切换点的代价

def mp_latency(bits, t8=T_INT8, t16=T_FP16, tr=T_REFORMAT):
    t = sum(t8 if b == 8 else t16 for b in bits)
    t += tr * sum(1 for a, b in zip(bits, bits[1:]) if a != b)
    return t

# 先看一眼 reformat 为什么重要：同样 2 层 FP16，连续 vs 散落
c_cont = [16, 16, 8, 8, 8, 8]
c_scat = [16, 8, 16, 8, 8, 8]
print('连续 2 层 FP16 %s -> %.2f ms' % (c_cont, mp_latency(c_cont)))
print('散落 2 层 FP16 %s -> %.2f ms  (**贵 %.2f ms，纯粹是 reformat**)'
      % (c_scat, mp_latency(c_scat), mp_latency(c_scat) - mp_latency(c_cont)))
assert mp_latency(c_scat) > mp_latency(c_cont)

ALL = [list(c) for c in itertools.product([8, 16], repeat=L_NET)]
table = [(mp_latency(c), net_acc(c), c) for c in ALL]
print()
print('%9s %10s %10s %s' % ('延迟预算', '最优精度', '延迟', '配置(8=INT8, 16=FP16)'))
best_by_budget = {}
for budget in [6.0, 7.0, 8.0, 9.0, 12.0]:
    feas = [(a, -t, c) for t, a, c in table if t <= budget + 1e-9]
    if not feas:
        print('%9.1f %10s' % (budget, '不可行')); continue
    a, nt, c = max(feas)
    best_by_budget[budget] = (a, -nt, c)
    print('%9.1f %10.4f %10.2f %s' % (budget, a, -nt, c))

assert best_by_budget[6.0][0] == ACC_INT8            # 预算只够全 INT8
assert best_by_budget[12.0][0] == ACC_FP16           # 预算充裕 -> 全 FP16
assert best_by_budget[8.0][0] > best_by_budget[6.0][0] + 0.05

# 贪心：从全 INT8 出发，每次挑「每毫秒收益最大」的层放回 FP16
def greedy(budget):
    bits = [8] * L_NET
    while True:
        cand = []
        for i in range(L_NET):
            if bits[i] == 16:
                continue
            b = list(bits); b[i] = 16
            if mp_latency(b) > budget + 1e-9:
                continue
            gain = net_acc(b) - net_acc(bits)
            cost = mp_latency(b) - mp_latency(bits)
            cand.append((gain / max(cost, 1e-9), i))
        if not cand:
            break
        g, i = max(cand)
        if g <= 0:
            break
        bits[i] = 16
    return net_acc(bits), mp_latency(bits), bits

print()
print('%9s %12s %12s %s' % ('延迟预算', '穷举最优', '贪心结果', '贪心配置'))
for budget in [7.0, 8.0, 9.0]:
    ga, gt, gb = greedy(budget)
    print('%9.1f %12.4f %12.4f %s' % (budget, best_by_budget[budget][0], ga, gb))
    assert best_by_budget[budget][0] >= ga - 1e-12, '穷举不可能输给贪心'
print()
print('✅ 贪心通常够用，但**不保证最优** —— 因为 reformat 代价让层与层之间产生了耦合')
print('   （放回相邻的两层比放回不相邻的两层便宜），这破坏了贪心所需的可分性。')
print('   实践建议：层数少时直接穷举；层数多时按「连续区间」枚举而不是按单层枚举。')

## 6 · 检测专属：回归头、小目标、以及 concat 的 scale 灾难

In [ ]:
def iou_shift(s, d):
    """边长 s 的正方形框，沿对角线整体位移 d 后与原框的 IoU"""
    inter = max(s - d, 0.0) ** 2
    return inter / (2.0 * s * s - inter)

print('手算校验：')
print('  8x8  框位移 2px : IoU = 36/92   = %.4f  <- **已低于 0.5 阈值，判为漏检**' % iou_shift(8, 2))
print('  8x8  框位移 1px : IoU = 49/79   = %.4f' % iou_shift(8, 1))
print('  64x64框位移 2px : IoU = 3844/4348 = %.4f  <- 几乎无感' % iou_shift(64, 2))
assert abs(iou_shift(8, 2) - 36 / 92) < 1e-12
assert abs(iou_shift(64, 2) - 3844 / 4348) < 1e-12
assert iou_shift(8, 2) < 0.5 < iou_shift(64, 2)
print()

def iou_mc(s, sigma, n=40000, seed=0):
    """四条边各加 **独立** N(0,sigma) 误差后的 IoU 分布。

    注意这与上面的 iou_shift 是**两个不同的误差模型**：
      · iou_shift(s, d)  = 整框沿对角线**整体位移** d  -> 误差单向累积，最悲观
      · iou_mc(s, sigma) = 四条边**独立**加噪         -> 约一半情况下互相抵消
    同样的数值下，独立模型温和得多（σ=1 时 8x8 还有 96%，而整体位移 1px 只剩 IoU 0.62）。
    **量化引入的坐标误差更接近独立模型**，所以估计敏感性、定容差时要用它；
    拿整体位移的直觉去套，会把危险高估一个档位。"""
    r = np.random.default_rng(seed)
    e = r.normal(0, sigma, (n, 4))
    x1, y1 = e[:, 0], e[:, 1]
    x2, y2 = s + e[:, 2], s + e[:, 3]
    iw = np.clip(np.minimum(x2, s) - np.maximum(x1, 0.0), 0, None)
    ih = np.clip(np.minimum(y2, s) - np.maximum(y1, 0.0), 0, None)
    inter = iw * ih
    ap = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / np.maximum(s * s + ap - inter, 1e-9)

SIZES = [8, 16, 32, 64, 128]
SIGMAS = [0.5, 1.0, 2.0, 3.0]
print('P(IoU >= 0.5)  —— 行=框边长(px)，列=量化引入的坐标误差 σ(px)')
print('%8s' % '边长', ''.join('%12s' % ('σ=%.2f' % g) for g in SIGMAS))
recall = {}
for s in SIZES:
    row = []
    for g in SIGMAS:
        p = float((iou_mc(s, g, seed=s * 10 + int(g * 100)) >= 0.5).mean())
        recall[(s, g)] = p; row.append(p)
    print('%8d' % s, ''.join('%11.3f ' % v for v in row))

assert recall[(8, 2.0)] < 0.60, recall[(8, 2.0)]              # 实测 ≈ 0.446
assert recall[(64, 2.0)] > 0.999, recall[(64, 2.0)]           # 实测 = 1.000
assert recall[(8, 2.0)] < recall[(64, 2.0)] - 0.30
assert recall[(8, 1.0)] < 0.999 and recall[(32, 1.0)] > 0.999
print()
print('⚠️  **同样 2 px 的坐标误差：8x8 框的召回掉到 %.1f%%，64x64 框仍是 %.1f%%。**'
      % (100 * recall[(8, 2.0)], 100 * recall[(64, 2.0)]))
print('   即便只有 1 px，8x8 也已经掉 %.1f%%，而 32x32 以上完全无感。'
      % (100 * (1 - recall[(8, 1.0)])))
print('   这就是「回归头必须优先保 FP16」的定量依据 —— 它没有任何排序容错。')
print()
print('⚠️  **别把两个误差模型混着用**：整体位移 1px 就让 8x8 的 IoU 掉到 %.3f，'
      % iou_shift(8, 1))
print('   而四边独立加噪 σ=1px 时 8x8 的召回还有 %.1f%% —— 独立误差会互相抵消。'
      % (100 * recall[(8, 1.0)]))
print('   量化引入的坐标误差接近**独立模型**；系统性偏移（letterbox 逆变换写错、')
print('   坐标系差半个像素）才是**位移模型**。定容差时用错模型会差一个档位。')

In [ ]:
# —— 分类头有多能扛？对比一下 ——
rc = np.random.default_rng(5)
scores = rc.beta(1.5, 6.0, 20000)                 # 检测器的分数分布：绝大多数很低
print('%10s %14s %16s' % ('分数噪声 σ', '过阈(0.3)翻转率', 'top-100 集合变动率'))
top100 = set(np.argsort(-scores)[:100].tolist())
for g in [0.005, 0.01, 0.03, 0.10]:
    noisy = np.clip(scores + rc.normal(0, g, scores.shape), 0, 1)
    flip = float(np.mean((scores >= 0.3) != (noisy >= 0.3)))
    t2 = set(np.argsort(-noisy)[:100].tolist())
    churn = 1 - len(top100 & t2) / 100
    print('%10.3f %13.3f%% %15.1f%%' % (g, 100 * flip, 100 * churn))
    if g == 0.03:
        flip03 = flip
        assert flip < 0.06, flip

print()
print('✅ 分类头的分数噪声 0.03（约满量程的 3%%）只造成 %.1f%% 的过阈翻转 ——' % (100 * flip03))
print('   因为它只影响**恰好落在阈值附近**的那一小撮候选，其余的排序纹丝不动；')
print('   而且翻转的都是低置信候选，对最终 AP 的影响远小于翻转率本身。')
print('⚠️  而回归头的 2 px 误差就能让 8x8 目标掉 %.0f%% 的召回。'
      % (100 * (1 - recall[(8, 2.0)])))
print('   **容错能力差一个数量级 —— 这是检测量化最重要的一条不对称。**')
print()

# —— 分尺寸汇总：整体指标怎么把小目标的塌方平均掉 ——
TSR_MIX  = {8: 0.34, 16: 0.31, 32: 0.20, 64: 0.10, 128: 0.05}   # TSR：绝大多数是小目标
COCO_MIX = {8: 0.06, 16: 0.12, 32: 0.24, 64: 0.31, 128: 0.27}   # 通用数据集
SIG = 2.0
print('%-14s %10s %10s' % ('尺寸桶', '召回', '相对无噪声'))
for s in SIZES:
    print('%-14s %10.3f %10.3f' % ('%dx%d' % (s, s), recall[(s, SIG)], recall[(s, SIG)] - 1.0))
tsr = sum(w * recall[(s, SIG)] for s, w in TSR_MIX.items())
coco = sum(w * recall[(s, SIG)] for s, w in COCO_MIX.items())
print()
print('按 TSR 尺寸分布加权的整体召回 : %.3f   （掉 %.1f 个点）' % (tsr, 100 * (1 - tsr)))
print('按 COCO 尺寸分布加权          : %.3f   （掉 %.1f 个点）' % (coco, 100 * (1 - coco)))
assert 1 - tsr > 2.5 * (1 - coco)
print()
print('⚠️  **同一个量化误差，在 COCO 上掉 %.1f 点、在 TSR 上掉 %.1f 点。**'
      % (100 * (1 - coco), 100 * (1 - tsr)))
print('   所以「别人家 INT8 只掉 0.5 mAP」这句话对 TSR 完全没有参考价值。')
print('   **TSR 的量化验收标准必须是「按像素尺寸分桶后最小的那一桶掉多少」，')
print('   外加「远处标志的首次检出距离退化多少米」。**')

In [ ]:
# —— 多尺度 concat 共用一个 per-tensor scale 的灾难 ——
rq = np.random.default_rng(17)
BRANCH = {}
for name, sc in [('P3 (stride 8)', 0.55), ('P4 (stride 16)', 1.7), ('P5 (stride 32)', 6.0)]:
    BRANCH[name] = np.abs(rq.normal(0, sc, 40000))
for k, v in BRANCH.items():
    print('%-16s max=%7.3f  99.9%%=%7.3f' % (k, v.max(), np.percentile(v, 99.9)))

cat = np.concatenate(list(BRANCH.values()))
thr_shared = percentile_threshold(cat, 99.9)
print('\nconcat 之后的共享阈值 = %.3f  （由 P5 决定）' % thr_shared)
print()
print('%-16s %16s %16s %10s' % ('分支', '共享 scale SQNR', '各自 scale SQNR', '差距'))
gaps = {}
for k, v in BRANCH.items():
    t_own = percentile_threshold(v, 99.9)
    a, _ = quant_dequant(v, thr_shared)
    b, _ = quant_dequant(v, t_own)
    d1, d2 = sqnr_db(v, a), sqnr_db(v, b)
    gaps[k] = d2 - d1
    print('%-16s %13.2f dB %13.2f dB %8.2f dB' % (k, d1, d2, d2 - d1))

assert gaps['P3 (stride 8)'] > 15, gaps
assert gaps['P3 (stride 8)'] > gaps['P5 (stride 32)'] + 10
print()
print('⚠️  **P3 被压掉了 %.1f dB ≈ %.1f 位有效精度**（P4/P5 几乎没损失）。'
      % (gaps['P3 (stride 8)'], gaps['P3 (stride 8)'] / 6.02))
print('   而 P3 正是小目标（远处交通标志）的特征来源 —— 又一次，')
print('   **量化的伤害精准地落在了最脆弱的那一层上。**')
print()
print('   解法：各分支先各自量化再 concat（显式 Q/DQ 图里完全可控），')
print('   或者训练时就用归一化把各层级的激活范围对齐（如 FPN 后加 LayerNorm/BN）。')
print('   隐式校准模式下你只能寄希望于 TRT 自己判断 —— 这是推荐显式量化的一个具体理由。')

## ✏️ 练习 1：per-channel 权重量化

实现 `per_channel_qdq(W, axis=1, n=127)`：沿 `axis` 求每个通道的 `max|w|` 作为阈值，
逐通道量化-反量化，返回与 `W` 同形状的结果。

（约定：`W` 的第 0 维是输出通道，`axis=1` 表示「对每一行单独定 scale」。）

In [ ]:
def per_channel_qdq(W, axis=1, n=127):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# (a) 每行的最大值应被**精确**还原（它正好落在第 127 个电平上）
A = np.array([[2.0, -1.0, 0.5],
              [200.0, 100.0, -50.0]])
Aq = per_channel_qdq(A)
assert Aq.shape == A.shape
assert abs(Aq[0, 0] - 2.0) < 1e-12, Aq           # 行 0 的 max=2   -> step=2/127
assert abs(Aq[1, 0] - 200.0) < 1e-12, Aq         # 行 1 的 max=200 -> step=200/127
# 其余元素的误差不得超过半个步长（步长是**逐行**的）
step = np.abs(A).max(axis=1, keepdims=True) / 127
assert np.all(np.abs(Aq - A) <= step / 2 + 1e-9), np.abs(Aq - A) - step / 2
# 对比 per-tensor：行 0 的元素被行 1 的量级绑架
At, _ = quant_dequant(A, np.abs(A).max())
print('per-channel 行0 =', Aq[0], '  误差 %.4f' % np.abs(Aq[0] - A[0]).max())
print('per-tensor  行0 =', At[0], '  误差 %.4f' % np.abs(At[0] - A[0]).max())
assert np.abs(Aq[0] - A[0]).max() < 0.1 * np.abs(At[0] - A[0]).max()
# (b) 在通道尺度极不均衡的矩阵上，per-channel 必须显著优于 per-tensor
rq2 = np.random.default_rng(3)
Wt = rq2.normal(0, 1, (32, 25)) * np.exp(rq2.normal(0, 1.1, (32, 1)))
pt, _ = quant_dequant(Wt, np.abs(Wt).max())
pc = per_channel_qdq(Wt)
print('per-tensor  %.2f dB' % sqnr_db(Wt, pt))
print('per-channel %.2f dB' % sqnr_db(Wt, pc))
assert sqnr_db(Wt, pc) > sqnr_db(Wt, pt) + 6
# (c) 通道尺度均衡时收益变小，但 **per-channel 永不更差**
#     （每行阈值 <= 全局阈值 -> 每行步长 <= 全局步长 -> 逐元素误差不可能更大）
We = rq2.normal(0, 1, (32, 25))
e_pc = sqnr_db(We, per_channel_qdq(We))
e_pt = sqnr_db(We, quant_dequant(We, np.abs(We).max())[0])
print('通道尺度均衡时: per-tensor %.2f dB -> per-channel %.2f dB  (只赚 %.2f dB)'
      % (e_pt, e_pc, e_pc - e_pt))
assert e_pc >= e_pt - 1e-9, (e_pt, e_pc)
assert (e_pc - e_pt) < (sqnr_db(Wt, pc) - sqnr_db(Wt, pt)), '不均衡时收益应更大'
print('✅ 练习 1 通过：per-channel **永不更差**，而收益大小完全取决于通道尺度有多不均衡 ——')
print('   所以在 depthwise / 分组卷积上它是硬要求，不是可选优化。')

## ✏️ 练习 2：熵校准的两个核心构造

实现：

- `reference_dist(hist, i)` → 参考分布 $P$：取 `hist[:i]`，把 `hist[i:]` 的**总质量并进最后一个 bin**
- `candidate_dist(hist, i, n_levels)` → 候选分布 $Q$：把 `hist[:i]` 均分成 `n_levels` 组
  （每组 `i // n_levels` 个 bin，**余数并入最后一组**），每组求和后**按组内非零 bin 的个数摊回去**，
  原本是 0 的 bin 保持 0

这两个函数就是 KL 校准的全部难点。

In [ ]:
def reference_dist(hist, i):
    # TODO
    raise NotImplementedError

def candidate_dist(hist, i, n_levels=128):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（全部可手算）——
hh = np.array([5, 3, 2, 1, 4], dtype=float)
P = reference_dist(hh, 3)
assert np.allclose(P, [5, 3, 2 + 1 + 4]), P          # 离群质量并进最后一个 bin
assert abs(P.sum() - hh.sum()) < 1e-12, '总质量必须守恒'
assert len(reference_dist(hh, 5)) == 5 and np.allclose(reference_dist(hh, 5), hh)

h2 = np.array([4, 0, 2, 6, 9, 9], dtype=float)
Q = candidate_dist(h2, 4, n_levels=2)                # 前 4 个 bin 压成 2 个电平
# 组1 [4,0] 和=4 非零=1 -> [4,0]；组2 [2,6] 和=8 非零=2 -> [4,4]
assert np.allclose(Q, [4, 0, 4, 4]), Q
assert abs(Q.sum() - h2[:4].sum()) < 1e-12
assert np.all((h2[:4] == 0) == (Q == 0)), '零 bin 必须保持零'

h3 = np.array([4, 0, 2, 0, 6, 0, 3, 3, 0, 1, 0, 0], dtype=float)
assert np.allclose(candidate_dist(h3, 12, 4), [3, 0, 3, 0, 6, 0, 3, 3, 0, 1, 0, 0])
# 余数并入最后一组：7 个 bin 压成 2 个电平 -> nm=3，组1=[0:3]，组2=[3:7]
h4 = np.array([2, 0, 4, 1, 1, 1, 1], dtype=float)
Q4 = candidate_dist(h4, 7, n_levels=2)
assert np.allclose(Q4, [3, 0, 3, 1, 1, 1, 1]), Q4

# 用它们复现第 3 节的 KL 扫描
hist_full, _ = np.histogram(np.abs(X), bins=2048, range=(0.0, float(np.abs(X).max())))
hist_full = hist_full.astype(float)
def kl_at(i):
    p = reference_dist(hist_full, i)
    q = candidate_dist(hist_full, i, 128)
    q = np.where(p == 0, 0.0, q)
    ps, qs = _smooth(p), _smooth(q)
    return kl_divergence(ps, qs)
assert kl_at(best_i) <= min(kl_at(best_i - 20), kl_at(best_i + 20), kl_at(2048))
print('best_i=%d  KL=%.3e   （左右各 20 个 bin 与末端的 KL 都更大）' % (best_i, kl_at(best_i)))
print('✅ 练习 2 通过：你已经把 TensorRT 默认校准算法的核心写出来了。')

## ✏️ 练习 3：IoU 对坐标误差的尺寸依赖

实现：

- `iou_shift2(s, d)` → 边长 `s` 的正方形沿对角线位移 `d` 后的 IoU：
  $\mathrm{IoU}=(s-d)^2/\bigl(2s^2-(s-d)^2\bigr)$（`d >= s` 时为 0）
- `min_box_size(d, target_iou=0.5)` → 使 `iou_shift2(s, d) >= target_iou` 的**最小整数** `s`（从 1 开始试）

In [ ]:
def iou_shift2(s, d):
    # TODO
    raise NotImplementedError

def min_box_size(d, target_iou=0.5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(iou_shift2(8, 0) - 1.0) < 1e-12
assert abs(iou_shift2(8, 1) - 49 / 79) < 1e-12
assert abs(iou_shift2(8, 2) - 36 / 92) < 1e-12
assert abs(iou_shift2(64, 2) - 3844 / 4348) < 1e-12
assert iou_shift2(8, 8) == 0.0 and iou_shift2(8, 20) == 0.0
assert min_box_size(0, 0.5) == 1
assert min_box_size(1, 0.5) == 6,  min_box_size(1, 0.5)     # 5:16/34=0.471 < 0.5 <= 6:25/47=0.532
assert min_box_size(2, 0.5) == 11, min_box_size(2, 0.5)     # 10:64/136=0.471 < 0.5 <= 11:81/161=0.503
assert min_box_size(2, 0.75) > min_box_size(2, 0.5)
print('%6s %14s %14s' % ('位移 d', '保住 IoU>=0.5', '保住 IoU>=0.75'))
for d in [0.5, 1, 2, 3, 4]:
    print('%6.1f %12d px %12d px' % (d, min_box_size(d, 0.5), min_box_size(d, 0.75)))
print()
print('✅ 练习 3 通过。TSR 落点：一块 60cm 的限速牌在 60m 外的 1080p/60°FOV 相机上约 20 px，')
print('   缩到 640 输入只剩 7 px。**此时 1 px 的量化误差就够把它判成漏检。**')

## ✏️ 练习 4：混合精度的延迟模型与预算选型

实现：

- `mp_latency2(bits, t8=1.0, t16=1.9, tr=0.2)` → 总延迟 = 各层耗时 + **每个精度切换点** `tr`
- `best_config(cands, budget)` → `cands` 是 `[(bits, acc), ...]`，
  返回预算内精度最高的 `(bits, acc, latency)`；都不可行时返回 `None`

In [ ]:
def mp_latency2(bits, t8=1.0, t16=1.9, tr=0.2):
    # TODO
    raise NotImplementedError

def best_config(cands, budget):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测（手算）——
assert abs(mp_latency2([8, 8, 8]) - 3.0) < 1e-12
assert abs(mp_latency2([16, 16, 16]) - 5.7) < 1e-12
assert abs(mp_latency2([8, 8, 16]) - 4.1) < 1e-12      # 1+1+1.9 + 1 个切换点
assert abs(mp_latency2([8, 16, 8]) - 4.3) < 1e-12      # 1+1.9+1 + 2 个切换点
assert mp_latency2([8, 16, 8]) > mp_latency2([8, 8, 16]), '同样 1 层 FP16，夹在中间更贵'

CAND = [([8, 8, 8], 0.62), ([8, 8, 16], 0.75), ([8, 16, 8], 0.71),
        ([16, 8, 8], 0.68), ([8, 16, 16], 0.88), ([16, 16, 16], 1.00)]
b, a, t = best_config(CAND, 4.2)
assert b == [8, 8, 16] and abs(a - 0.75) < 1e-12 and abs(t - 4.1) < 1e-12, (b, a, t)
b2, a2, t2 = best_config(CAND, 5.2)
assert b2 == [8, 16, 16] and abs(t2 - 5.0) < 1e-12, (b2, a2, t2)   # 1+1.9+1.9 + 1 个切换点
assert best_config(CAND, 2.0) is None
for budget in [2.0, 3.5, 4.2, 5.2, 9.0]:
    r = best_config(CAND, budget)
    print('预算 %.1f ms -> %s' % (budget, '不可行' if r is None else
                                  '%s  acc=%.2f  lat=%.2f' % (r[0], r[1], r[2])))
print('✅ 练习 4 通过：**reformat 代价让「哪几层」比「几层」更重要** ——')
print('   同样一层 FP16，放在两端只加 1 个切换点，夹在中间要加 2 个。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def per_channel_qdq(W, axis=1, n=127):
    W = np.asarray(W, dtype=float)
    thr = np.abs(W).max(axis=axis, keepdims=True)
    s = np.where(thr > 0, thr / n, 1.0)
    return np.clip(np.round(W / s), -n, n) * s

In [ ]:
# 练习 2 参考答案
def reference_dist(hist, i):
    hist = np.asarray(hist, dtype=float)
    p = hist[:i].copy()
    p[-1] += hist[i:].sum()
    return p

def candidate_dist(hist, i, n_levels=128):
    h = np.asarray(hist, dtype=float)[:i]
    nm = i // n_levels
    starts = np.arange(n_levels) * nm
    sums = np.add.reduceat(h, starts)
    nzc = np.add.reduceat((h > 0).astype(float), starts)
    gid = np.repeat(np.arange(n_levels), np.diff(np.append(starts, i)))
    return np.where(h > 0,
                    np.where(nzc[gid] > 0, sums[gid] / np.maximum(nzc[gid], 1.0), 0.0),
                    0.0)

In [ ]:
# 练习 3 参考答案
def iou_shift2(s, d):
    inter = max(s - d, 0.0) ** 2
    return inter / (2.0 * s * s - inter)

def min_box_size(d, target_iou=0.5):
    s = 1
    while iou_shift2(s, d) < target_iou:
        s += 1
        if s > 100000:
            raise ValueError('无解')
    return s

In [ ]:
# 练习 4 参考答案
def mp_latency2(bits, t8=1.0, t16=1.9, tr=0.2):
    t = sum(t8 if b == 8 else t16 for b in bits)
    return t + tr * sum(1 for a, b in zip(bits, bits[1:]) if a != b)

def best_config(cands, budget):
    feas = [(a, -mp_latency2(b), b) for b, a in cands if mp_latency2(b) <= budget + 1e-9]
    if not feas:
        return None
    a, nt, b = max(feas)
    return b, a, -nt

---
## 🧪 真实工程胶囊：校准脚本模板 + INT8 验收标准

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 校准集构造（80% 的 INT8 掉点问题在这一步，先把它做对）
# ══════════════════════════════════════════════════════════════════════
# 1) **复用线上的预处理函数，不许重写一份**
#    from deploy.preprocess import preprocess   # <- 与 C++ 端对齐过的那一份
#    禁止使用 train_loader（它带着 Mosaic / 色彩抖动 / RandomErasing）
#
# 2) 分层采样：按部署分布配额 + 每个关键场景设下限 + 每片段最多取 1~2 帧
#    QUOTA = {                      # 目标 ~800 张
#      'day_clear':   360,          # 按部署分布
#      'dusk':        100,
#      'night':       140,          # **反光膜 + 车灯 -> 激活极值的主要来源**
#      'rain_snow':    80,
#      'tunnel':       60,          # **出入口的极端动态范围，必须有配额**
#      'backlit':      60,          # 逆光
#    }
#    规则：每个片段(sequence)最多取 2 帧；同一路段同一时段最多取 N 帧
#
# 3) 自检（写成脚本，进 CI）：
#    · 独立片段数 >= 0.5 * 帧数        （防连续帧）
#    · 每个场景标签的实际张数 >= 配额的 80%
#    · **预处理指纹**：对同一张图，Python 校准链路与 C++ 部署链路的张量
#      逐元素 max|diff| < 1e-6        （模块 01 的对拍工具）
#    · 校准集里不能出现任何增强产物（拼图/纯色块/异常色相）

# ══════════════════════════════════════════════════════════════════════
# B. 算法选择：三个都跑，用分桶评测决定（各只要几十分钟）
# ══════════════════════════════════════════════════════════════════════
# TensorRT 隐式量化：
#   IInt8EntropyCalibrator2   <- CNN 检测/分类的默认首选
#   IInt8MinMaxCalibrator     <- **BERT/Transformer 类反而更好**
# 显式量化（推荐，TRT 8.x+）：用 pytorch-quantization / TensorRT Model Optimizer
#   产出带 QuantizeLinear/DequantizeLinear 的 ONNX，量化点由你决定
#
# trtexec 侧：
#   trtexec --onnx=det.onnx --int8 --fp16 --calib=calib.cache \
#           --saveEngine=det_int8.plan --precisionConstraints=obey \
#           --layerPrecisions=head.reg.*:fp16,backbone.stem.*:fp16
#
# calibration cache 的归档规则（缺一不可）：
#   calib.cache + {preprocess_version, weights_sha256, calibset_manifest_sha256,
#                  calibrator_type, n_images, scene_quota}
#   **cache 可以跨 GPU 型号复用；不能跨预处理复用。**

# ══════════════════════════════════════════════════════════════════════
# C. 掉点排查顺序（照这个顺序走，不要跳步）
# ══════════════════════════════════════════════════════════════════════
#  0. FP16 engine 的指标 == PyTorch 吗？不等 -> 不是量化问题（回模块 01/04）
#  1. 校准集：预处理一致？关增强了？场景覆盖？独立片段数？大小扫描 128/512/1024/2048
#  2. 校准算法：Entropy / Percentile(99.9, 99.99, 99.999) / MinMax 各建一个 engine
#  3. 逐层 SQNR（FP32 vs INT8 同层输出），标出 < 20 dB 的层；再用「恢复方向」确认
#     结构性检查：depthwise 是否 per-channel / concat 是否共用 scale /
#                 残差 Add 两路 scale 差多少 / Reformat 层数量是否异常
#  4. 混合精度：**按连续块划分**，画「FP16 层数 - 精度 - 延迟」曲线
#  5. 仍不行才上 QAT（1~3 周，且需要完整训练管线；BN 必须先折叠再插 fake-quant）

# ══════════════════════════════════════════════════════════════════════
# D. TSR 的 INT8 验收标准（不要用「整体 mAP 掉 < 1」这种口径）
# ══════════════════════════════════════════════════════════════════════
#  必须按**像素尺寸**分桶：  <16px / 16-32 / 32-64 / >64
#  必须按**场景**分桶：      day / dusk / night / rain / tunnel / backlit
#  门禁建议：
#    · 最小尺寸桶的 AP 退化   < 1.5      （整体 mAP 会把它平均掉）
#    · 每个场景桶的 AP 退化   < 1.0      （夜间/隧道是重灾区）
#    · **首次检出距离退化     < 5 m**    （TSR 的核心产品指标）
#    · 关键类别（停车让行/限速）的召回退化 < 0.5
#    · 延迟 p99 相对 FP16 的收益 > 25%   （否则不值得承担量化风险）
#  上线后监控：
#    · 车端记录关键层的**裁剪率**并按场景标签统计
#    · 某场景裁剪率显著高于校准时的水平 = 校准集缺这个场景
#      （这是不需要标注、不需要重评测的早期告警信号）
'''
print(RECIPE)
for token in ['train_loader', 'IInt8EntropyCalibrator2', 'IInt8MinMaxCalibrator',
              'precisionConstraints=obey', '独立片段数', '裁剪率',
              '首次检出距离', 'BN 必须先折叠', 'preprocess_version']:
    assert token in RECIPE, token
print('✅ 覆盖：校准集构造与自检 / 算法选择 / cache 归档 / 排查顺序 / TSR 验收门禁 / 上线监控')

### 小结

- **对称量化 + per-tensor 激活 + per-channel 权重**，这不是三个独立选择，而是一条代数结论：
  权重 scale 与求和变量无关所以能提到 INT32 累加之外，激活 scale 落在求和号内所以不能。
  **SmoothQuant 正是利用这个不对称，把激活的通道差异「搬」到权重上去。**
- **MinMax 会把 8 位量化器变成 2.5 位**（本例：主体只占 127 个电平里的 5.5 个，SQNR 17 dB）。
  KL 熵校准把它救到 6.2 位。但 **Percentile 的分位点必须与离群比例匹配** ——
  本例里 99.99% 比 99.9% 差了 22.8 dB，比 MinMax 还糟。
- **KL 校准的完整算法**：2048-bin 直方图 → 从第 128 个 bin 起扫描截断点 →
  离群质量并进最后一个 bin 得参考分布 P → 压成 128 级再按非零 bin 摊回得候选分布 Q →
  取 KL 最小 → threshold = (i*+0.5)×bin 宽。**它只在有离群时才裁剪**；
  离群多到 0.5% 时它又回到 MinMax 附近，因为那已经不是离群了。
- **一个必须记住的反直觉**：按全张量 SQNR 评判 MinMax 反而赢（18.9 vs 6.1 dB），
  按主体 SQNR KL 赢 22 dB。**「哪个算法好」依赖于你认为哪部分信息重要，
  这只能由下游任务指标回答** —— 所以必须落到分桶评测。
- **校准集才是主要矛盾**。同样 512 张图、同样算法：只用白天 → 夜间裁剪率 49%、准确率掉 23 个点；
  分层采样 → 完全无损。「训练集头 500 张」叠了三个错误：只有 25 个独立片段、
  尾部场景覆盖 0%、还带着训练增强。
- **两个扫描方向不等价**（孤立量化 vs 恢复到 FP16），甚至会出现「恢复某层反而略降」——
  量化误差在层间会相互抵消。**做方案用「恢复」方向，且必须整体验证，不能逐层加总。**
  混合精度要**按连续块划分**：同样 2 层 FP16，散落比连续多付 2 个 reformat。
- **检测的特殊性**：分类头的分数噪声 0.03 只造成 ~1% 过阈翻转，
  而回归头 2 px 的误差就让 8×8 目标掉 55% 召回（64×64 几乎无感）。
  多尺度 concat 共用 per-tensor scale 会把 P3 压掉 15+ dB（约 2.6 位）—— 而 P3 正是小目标的特征来源。
  **同一个 2 px 坐标误差，按 COCO 尺寸分布掉 3.8 点召回，按 TSR 分布掉 20 点。**
- **排查顺序比方法更重要**：先确认不是一致性问题 → 校准集 → 算法 → 敏感层 →
  混合精度 → 最后才是 QAT。**直接上 QAT 是面试减分项**，因为 80% 的掉点在校准集上半天能解决。

下一站：**模块 04 · 后处理对齐与 C++ 推理管线** —— 量化会让分数出现大量并列，
NMS 的排序稳定性问题就从这里开始。